In [0]:
import logging
logging.getLogger("pyspark.sql.connect.logging").setLevel(logging.ERROR)
import time
import numpy as np
import pandas as pd
import mlflow
from datetime import datetime, timezone

t0 = time.time()
run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
HALF_LIFE = 1.0   # months; chosen Oct 13 2026 by lowest pre-war interval score (see 42_forecast_comparison)

# ---------- EIA track record vs no-change ----------
actual = spark.sql("""
    SELECT series_id, date_trunc('month', price_date) AS month, AVG(price) AS actual
    FROM oil_shock.gold.fct_price_daily
    WHERE series_id IN ('RBRTE', 'RWTC') AND price_date < date_trunc('month', current_date())
    GROUP BY series_id, date_trunc('month', price_date)
""").toPandas()
actual["month"] = pd.to_datetime(actual["month"])
actual["series"] = actual["series_id"].map({"RBRTE": "BREPUUS", "RWTC": "WTIPUUS"})
a = actual[["series", "month", "actual"]]

steo = spark.sql("""
    SELECT release_month, series_id, target_month, value AS forecast,
           CAST(ROUND(months_ahead) AS INT) AS horizon, source
    FROM oil_shock.gold.fct_steo_forecast
    WHERE series_id IN ('BREPUUS', 'WTIPUUS') AND is_forecast
""").toPandas()
steo["target_month"] = pd.to_datetime(steo["target_month"])
steo["release_date"] = pd.to_datetime(steo["release_month"] + "-01")
steo = steo[steo["horizon"].between(0, 12)]

df = steo.merge(a, left_on=["series_id", "target_month"], right_on=["series", "month"], how="inner")
df["base_month"] = df["release_date"] - pd.offsets.MonthBegin(1)
df = df.merge(a.rename(columns={"month": "base_month", "actual": "baseline"}),
              left_on=["series_id", "base_month"], right_on=["series", "base_month"], how="left", suffixes=("", "_b"))
df = df.dropna(subset=["baseline"])
df["steo_err_pct"] = 100 * (df["forecast"] - df["actual"]) / df["actual"]
df["base_err_pct"] = 100 * (df["baseline"] - df["actual"]) / df["actual"]
df["steo_better"] = df["steo_err_pct"].abs() < df["base_err_pct"].abs()
df["period"] = np.where(df["target_month"] >= "2026-03-01", "war", "pre_war")

summary = (df.groupby(["series_id", "period", "horizon"])
           .agg(n=("steo_err_pct", "size"),
                steo_mape=("steo_err_pct", lambda x: x.abs().mean()),
                baseline_mape=("base_err_pct", lambda x: x.abs().mean()),
                steo_bias=("steo_err_pct", "mean"),
                baseline_bias=("base_err_pct", "mean"),
                steo_beats_baseline_pct=("steo_better", lambda x: 100 * x.mean()))
           .reset_index().round(1))

detail = df[["series_id", "release_month", "target_month", "horizon", "period", "source",
             "forecast", "actual", "baseline", "steo_err_pct", "base_err_pct", "steo_better"]].copy()
detail["target_month"] = detail["target_month"].dt.date

# ---------- Current 80% range around the latest EIA release ----------
fv = spark.table("oil_shock.gold.fct_model_features_daily").select("price_date", "brent_vol_20d").toPandas()
fv["price_date"] = pd.to_datetime(fv["price_date"])
fv = fv.dropna().sort_values("price_date")
v_long = fv.loc[fv["price_date"] < "2026-01-01", "brent_vol_20d"].median()

tr = detail[detail["series_id"] == "BREPUUS"].copy()
rel = pd.DataFrame({"release_month": tr["release_month"].unique()})
rel["day10"] = pd.to_datetime(rel["release_month"] + "-10")
rel = pd.merge_asof(rel.sort_values("day10"), fv, left_on="day10", right_on="price_date", direction="backward")
tr = tr.merge(rel[["release_month", "brent_vol_20d"]].rename(columns={"brent_vol_20d": "v"}), on="release_month")

def eff_vol(v, h):
    w = 0.5 ** (h / HALF_LIFE)
    return np.sqrt(w * v ** 2 + (1 - w) * v_long ** 2)

tr["z"] = np.log(tr["actual"] / tr["forecast"]) / eff_vol(tr["v"].values, tr["horizon"].values)
q = tr.groupby("horizon")["z"].quantile([0.1, 0.9]).unstack()

latest = steo["release_month"].max()
cur = steo[(steo["series_id"] == "BREPUUS") & (steo["release_month"] == latest)][
    ["release_month", "horizon", "target_month", "forecast"]].rename(columns={"forecast": "eia_forecast"})
cur = cur.sort_values("horizon").reset_index(drop=True)
v_now = fv.loc[fv["price_date"] <= pd.Timestamp(latest + "-10"), "brent_vol_20d"].iloc[-1]
ev = eff_vol(v_now, cur["horizon"].values)
cur["low_80"] = (cur["eia_forecast"] * np.exp(q.loc[cur["horizon"], 0.1].values * ev)).round(2)
cur["high_80"] = (cur["eia_forecast"] * np.exp(q.loc[cur["horizon"], 0.9].values * ev)).round(2)
cur["half_life_months"] = HALF_LIFE
cur["target_month"] = cur["target_month"].dt.date

for nm, out in {"steo_track_record": detail, "steo_accuracy_summary": summary, "steo_range_current": cur}.items():
    out = out.copy(); out["run_at_utc"] = run_stamp
    spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"oil_shock.ml.{nm}")

mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-forecast-comparison")
with mlflow.start_run(run_name=f"eia_range_refresh {run_stamp}"):
    mlflow.set_tags({"notebook": "notebooks/04_ml/46_eia_range_refresh", "project": "oil-shock-2026", "author": "Zia Malik"})
    mlflow.log_params({"latest_release": latest, "half_life_months": HALF_LIFE})
    mlflow.log_metric("track_record_rows", len(detail))

print(f"Done in {time.time() - t0:.0f}s | track record rows: {len(detail)} | latest release: {latest}")
print(cur[["target_month", "horizon", "eia_forecast", "low_80", "high_80"]].head(4).to_string(index=False))

In [0]:
# Weekly guard: in the scheduled job, run only on Monday's morning run (Dubai time).
# For a manual run on another day, set force_weekly = true.
from datetime import datetime
from zoneinfo import ZoneInfo

dbutils.widgets.dropdown("force_weekly", "false", ["false", "true"])
_now = datetime.now(ZoneInfo("Asia/Dubai"))
if not ((_now.weekday() == 0 and _now.hour < 12) or dbutils.widgets.get("force_weekly") == "true"):
    dbutils.notebook.exit(f"Skipped: weekly model, runs Monday mornings (now {_now:%a %H:%M} Dubai)")